# Breed Standard Text Analysis: Dimensionality Reduction Pipeline
Author: Halie M. Rando
Date: July 2025, updated January 2026

This pipeline demonstrates a proof-of-concept analysis to incorporating proscriptive text descriptions of breed as a ground truth for fine-grained image classification of dog images. 

This notebook outlines the steps needed to download, digest, analyze and interpret breed traits as specified in by the Fédération Cynologique Internationale, the breed standards used by most of the world. Breed standards are specifications of dog morphology (and sometimes other traits) used to assess conformation, especially in the context of dog shows and purebred dog registration.

References:
- Standard documentation for the packages
- Anthropic's Claude and OpenAI's ChatGPT were used to generate and critique code -- all code has been reviewed and tested by a human (HMR)

In [23]:
import re
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import sqlite3
from utils import extract_pdf_text_with_pymupdf, clean_and_fix_pdf_text

# import spacy
# import gensim
# import matplotlib.pyplot as plt
# 
# from sentence_transformers import SentenceTransformer
# from sklearn.tree import DecisionTreeClassifier, plot_tree
# from sklearn.feature_extraction.text import TfidfVectorizer
# from scipy.stats import spearmanr


In [24]:
# Select which breeds to analyze and specify URL of breed standard
# I previously scraped the FCI website in full -- only three URLs are provided here for demo purposes

# Here, I have chosen three breeds from the same breed group (FCI Group 8) that have a priori major differences: Labs and Goldens occupy different (but partially overlapping) color ranges, and Newfoundlands are much larger
breed_url = {"Labrador": "https://www.fci.be/en//Nomenclature/Standards/122g08-en.pdf",
             "Newfoundland": "https://www.fci.be/en//Nomenclature/Standards/050g02-en.pdf",
             "Golden Retriever": "https://www.fci.be/en//Nomenclature/Standards/111g08-en.pdf"}

# Try to load from disk if possible
check_db = True

In [25]:
# Load text from PDFs (retrieve from web based on URLs specified above)
breed_docs = []
breed_order = []
for breed, url in breed_url.items():
    cleaned_text = extract_pdf_text_with_pymupdf(url)
    cleaned_text = clean_and_fix_pdf_text(cleaned_text)
    print(cleaned_text)
    print("\n\n")
    breed_docs.append(cleaned_text)
    breed_order.append(breed)

federation cynologique internationale aisbl secretariat general 13 place albert 1er b  6530 thuin belgique ______________________________________________________________________________ 30092022 en fcistandard n 122 labrador retriever mdavidson illustr nku picture library  2 origin great britain date of publication of the official valid standard 16062022 utilization retriever fciclassification group 8 retrievers flushing dogs water dogs section 1 retrievers with working trial brief historical summary it is popularly thought that the labrador retriever originated on the coast of newfoundland where fishermen were seen to use a dog of similar appearance to retrieve fish an excellent water dog his weatherresistant coat and unique tail likened to that of an otter because of its shape emphasise this trait comparatively speaking the labrador is not a very old breed its breed club having been formed in 1916 and the yellow labrador club having been founded in 1925 it was in field trialling that

In [20]:
print(breed_docs )

['server error 403  forbidden access is denied you do not have permission to view this directory or page using the credentials that you supplied', 'federation cynologique internationale aisbl secretariat general 13 place albert 1er b  6530 thuin belgique ______________________________________________________________________________ 06111996en fcistandard n 50 newfoundland 2 origin canada patronage fci date of publication of the official valid standard 29101996 utilization sledge dog for heavy loads water dog fciclassification group 2 pinscher and schnauzer molossoid breeds swiss mountain and cattle dogs section 22 molossoid breeds mountain type without working trial short historical summary the breed originated in the island of newfoundland from indigenous dogs and the big black bear dog introduced by the vikings after the year 1100 with the advent of european fishermen a variety of new breeds helped to shape and reinvigorate the breed but the essential characteristics remained when th